# Lab 08a · When a model cannot read the page

[Open in Colab](https://colab.research.google.com/github/AIGuruOne/OQ-labs/blob/main/notebooks/08a_vision_basics.ipynb)

**Day 3 · S16 · Step 1 of 2** · 15 min · One API key. No GPU, no Ollama, nothing to install but one package.

## What you will be able to do after this

**Recognise that a vision model which cannot read a field returns a confident,
well-formed, wrong value rather than an error — and know that telling it to flag
uncertainty does not fix it.**

**You will know it landed** when you can answer this, out loud:

> *"Why is `PTW-48715` worse than a blank field?"*

**Where you use it:** step 2 (lab 09a) turns this into a rule you can put in your
capstone. S29 governance pack, template 3 — what the model may be trusted to do.

---

One work order. Three photographs of it. Three prompts. Nothing else.

**A correct result looks like:** the clean scan reads perfectly, the poor photograph
comes back **wrong with no error attached**, the obvious fix changes nothing, and only
an explicit instruction to assume it cannot read makes the model back off.

> All data in this lab is synthetic. No real OQ material anywhere.

## 1. Setup

One import. Everything mechanical lives in `vision_lab.py`.

In [ ]:
import sys, subprocess
if "google.colab" in sys.modules:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "openai==3.0.0"], check=True)
    subprocess.run(["git", "clone", "-q", "https://github.com/AIGuruOne/OQ-labs.git"], check=False)
    sys.path.insert(0, "OQ-labs/notebooks")
    import os
    from google.colab import userdata
    os.environ.setdefault("OPENAI_API_KEY", userdata.get("OPENAI_API_KEY"))

import vision_lab as lab
print("ready |", lab.MODEL)

## 2. One work order, and what is on it

We drew this page, so we know every value on it. That is the answer key for everything below.

In [ ]:
forms, truths = lab.make_forms(1, level="clean")
TRUTH = truths[0]
TRUTH

In [ ]:
lab.show(forms[0])

## 3. Ask the model to read it

A plain prompt. Six keys. Nothing clever.

In [ ]:
answer = lab.read_form(forms[0], "plain")
print(lab.compare_one(answer, TRUTH))

Six out of six. **This is where the false confidence comes from:** on a good scan a
current vision model reads a form like this perfectly, every time.

## 4. Now photograph it

Nobody sends a flatbed scan from the field. Same page, taken on a phone in a dim plant
room from too far back.

In [ ]:
poor = lab.photograph(lab.draw_form(TRUTH), "poor")
lab.show(poor)

In [ ]:
print(lab.compare_one(lab.read_form(poor, "plain"), TRUTH))

## 5. Stop and look at *how* it failed

Not "the model failed". Look at what came back.

- The JSON is **valid**.
- Every value is the **right shape** — a permit that looks like a permit number, a date
  that looks like a date.
- A permit of `PTW-48213` comes back as something like `PTW-48715`: same prefix, same
  length, different number.
- There is **no error, no warning, no confidence score, no blank field.**

Everything a normal pipeline checks for would pass. A person reviewing this output,
without the photograph beside it, has no reason to doubt any of it. And permits control
who is allowed to do dangerous work.

## 6. The fix everybody writes first

Tell it to say so when it is unsure. This is the instruction almost everyone writes, and
it would pass a code review.

In [ ]:
print(lab.compare_one(lab.read_form(poor, "soft"), TRUTH))

**It changes nothing.** Same wrong values, still nothing flagged.

This is the most useful cell in the lab. *Asking politely is not a control.*

## 7. What does work

Invert the default: assume the image is unreadable, make refusal the norm, and name
identifiers as the highest-risk fields.

In [ ]:
print(lab.compare_one(lab.read_form(poor, "careful"), TRUTH))

### And check it has not just become useless

A model that refuses everything is no better than one that guesses everything.

In [ ]:
print(lab.compare_one(lab.read_form(forms[0], "careful"), TRUTH))

## 8. What you take to step 2

**A vision model does not read a form. It interprets one.** On a good image the
interpretation is right; on a bad one it is confident, well-formed and wrong — and
nothing in the output separates the two.

**The failure carries no error.** No exception, no blank, no score. That is what makes
it dangerous against a system of record.

**Refusing has a price.** The careful prompt refuses fields it could have read. That is
a trade you choose, not a bug: a refusal is someone re-photographing a page, a wrong
permit number is someone working under the wrong permit.

So the open question, which is exactly **lab 09a**:

> *You cannot check every field by hand. Which ones must a person see?*

In [ ]:
print("08a done - a wrong value here is well-formed, confident, and unflagged.")